In [ ]:
# 1. Встановлюємо залежності з requirements.txt
# Шлях залежатиме від того, як ти назвеш датасет при завантаженні через "Add Input -> Upload".
# Заміни 'my-requirements' на свою назву.
!pip install -r /kaggle/input/my-requirements/requirements.txt

import wandb
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

# 2. Ініціалізація Weights & Biases для бонусних балів
# На Kaggle зручно зберігати токен у секретах (Add-ons -> Secrets -> додати WANDB_API_KEY).
# Це дозволить логінитись без ручного введення щоразу.
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
except:
    print("Секрет WANDB_API_KEY не знайдено. Буде запитано ручне введення.")
    wandb.login()

# 3. Беремо легку anchor-free модель YOLOv8n (nano)
model = YOLO('yolov8n.pt') 

# 4. Шлях до конфігу з даними (тягнеться з аутпуту першого ноутбука)
# УВАГА: Заміни 'eda-output' на реальну назву, яку Kaggle присвоїть підключеним результатам
# першого ноутбука (це видно у правому меню в розділі Data).
data_yaml_path = '/kaggle/input/eda-output/masked_dataset/data_masked.yaml'

# 5. Запуск тренування
results = model.train(
    data=data_yaml_path,
    epochs=10,
    imgsz=1024,
    batch=16, 
    device=[0, 1], # [0, 1] задіює обидві відеокарти T4 для максимальної швидкості
    project='/kaggle/working/ucu_ai_hw2', # Обов'язково зберігаємо в /kaggle/working/ щоб мати змогу завантажити ваги
    name='yolov8n_baseline'
)